[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-12-llm-context.ipynb)


# Lecture 12: does context help an LLM translate a language it barely knows?

The goal of this assignment is to explore how providing in-context examples affects the
translation quality of a large language model (LLM) for a low-resource language of your choice:
you show it a few translated sentences in the prompt, then more, and measure whether its
translations improve.

By the end you will be able to:

1. keep the sentences you show the model apart from the ones you test it on, and say why;
2. build a translation prompt with *k* example pairs in it, and say what "adding context" changes;
3. translate the same 50 test sentences at context sizes 0, 5, 10 and 20, score each with BLEU
   and its signature, and explain the curve against your own prediction.

You write the two functions that are the method, marked TODO; checks tell you when each is
right, and the notebook runs the rest. It writes everything Assignment 12 asks you to upload to
`MyDrive/CS479/assignment12`.

**Running in Google Colab?** Runtime → Change runtime type → a GPU, then Run all; it stops at
each TODO until you write it. It runs on a built-in English–Spanish sample until you change two
lines in Step 1 for your language.


<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — homework for Lecture 12 (*Using LLMs for MT; Expanding Context Awareness*). A head start on **A12**.

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo


In [ ]:
# Check the setup and connect Google Drive, where this notebook saves what you upload.
from torchlingo.colab import setup

env = setup(gpu=True, drive=True)       # stops with instructions if there is no GPU
device = env.device
OUT_DIR = env.drive_dir / "CS479" / "assignment12"
OUT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import torch


## Step 1: your language's sentences, split into context and test

Two sets, kept apart. The **context pool** supplies the example pairs in the prompt; the **test
set** is what the model translates and BLEU scores. A sentence in both would let the model copy
its answer out of the prompt.

For your language, set `LANGUAGE`, and set `DATA_DIR` to your language's folder from the A12
shared folder, copied into your Drive. The loader reads one tab-separated file, English first,
or two line-aligned text files with the English file's name containing `en`.

**Yours to write: `split_pool_and_test`.** The cell checks it on six pairs whose split you can
see, then runs the library's contamination check on your real split before saving it.


In [ ]:
from pathlib import Path

from torchlingo.diagnostics import check_contamination

LANGUAGE = "Spanish"    # <- your A12 language: Efik, Palauan, Pohnpeian, Yapese, Kosraean or Kamba
DATA_DIR = None         # <- your language's folder, e.g. env.drive_dir / "CS479" / "a12-data"

SEED = 479
POOL_SIZE, TEST_SIZE = 20, 50   # the largest context size, and 50 test sentences


def load_pairs(data_dir):
    """Return a DataFrame of English (src) and target-language (tgt) pairs from data_dir."""
    data_dir = Path(data_dir)
    tables = sorted(data_dir.glob("*.tsv"))
    texts = sorted(data_dir.glob("*.txt"))
    if len(tables) == 1:
        frame = pd.read_csv(tables[0], sep="\t", dtype=str, keep_default_na=False)
        frame = frame.iloc[:, :2]
    elif len(texts) == 2:
        english = [p for p in texts if "en" in p.stem.lower().replace("sentences", "")]
        if len(english) != 1:
            raise SystemExit(f"Cannot tell which of {[p.name for p in texts]} is English.")
        other = [p for p in texts if p != english[0]][0]
        src = english[0].read_text(encoding="utf-8").splitlines()
        tgt = other.read_text(encoding="utf-8").splitlines()
        if len(src) != len(tgt):
            raise SystemExit(f"{english[0].name} has {len(src)} lines, {other.name} {len(tgt)}.")
        frame = pd.DataFrame({"src": src, "tgt": tgt})
    else:
        raise SystemExit(
            f"In {data_dir}: expected one .tsv file or two .txt files, found "
            f"{[p.name for p in data_dir.iterdir()]}. Ask the teaching staff."
        )
    frame.columns = ["src", "tgt"]
    frame = frame[(frame.src.str.strip() != "") & (frame.tgt.str.strip() != "")]
    return frame.drop_duplicates("src").reset_index(drop=True)


if DATA_DIR is None:
    from torchlingo.colab import fetch_data

    (sample,) = fetch_data(["data/example.tsv"])
    pairs = pd.read_csv(sample, sep="\t", dtype=str, keep_default_na=False)
    words = pairs.src.str.split().str.len()
    pairs = pairs[(pairs.kind == "transcript") & words.between(6, 20)][["src", "tgt"]]
    pairs = pairs.drop_duplicates("src").reset_index(drop=True)
    print("*** The built-in English-Spanish sample. Spanish is not the assignment: set LANGUAGE")
    print("*** and DATA_DIR above to your A12 language, then run all again. ***\n")
else:
    pairs = load_pairs(DATA_DIR)
assert len(pairs) >= POOL_SIZE + TEST_SIZE, f"only {len(pairs)} pairs; need {POOL_SIZE + TEST_SIZE}"


def split_pool_and_test(pairs, pool_size, test_size, seed):
    """Return (pool, test): pool_size pairs to draw examples from, test_size pairs to translate.

    TODO: write it. Draw pool_size + test_size pairs at random, the same ones every time for the
    same seed (pandas' sample takes random_state), and give the first pool_size to the pool and
    the rest to the test set. No sentence may be in both.
    """
    raise NotImplementedError("write split_pool_and_test, then run this cell again")


# A case whose answer you know: six distinct pairs into two and three, the same way twice.
toy = pd.DataFrame({"src": list("abcdef"), "tgt": list("ABCDEF")})
toy_pool, toy_test = split_pool_and_test(toy, 2, 3, seed=0)
assert (len(toy_pool), len(toy_test)) == (2, 3), "the pool and test set are not the sizes asked for"
assert not set(toy_pool.src) & set(toy_test.src), "a pair is in both the pool and the test set"
assert split_pool_and_test(toy, 2, 3, seed=0)[1].equals(toy_test), "the same seed must give the same split"

pool, test = split_pool_and_test(pairs, POOL_SIZE, TEST_SIZE, SEED)
result = check_contamination(test.src, pool.src, name="test")
print(result)
assert result, "a test sentence is also in the context pool, so the model could copy its answer"

test.to_csv(OUT_DIR / "test.tsv", sep="\t", index=False)
print(f"{len(pairs):,} {LANGUAGE} pairs; {len(pool)} in the context pool, {len(test)} in the test set.")
print(f"Saved {OUT_DIR / 'test.tsv'}")


## Step 2: the prompt

"Adding context" means putting *k* English sentences and their translations into the prompt,
above the sentence to translate. At *k* = 0 the model has only the instruction and whatever it
learned about your language in pretraining. Each size takes the first *k* pairs of the pool, so
the 5 are part of the 10 and the 10 part of the 20: only how many the model sees changes.

**Yours to write: `build_prompt`.** The checks confirm what it must contain; the wording is
yours. The cell then prints your prompt at *k* = 5: everything the model knows about the task.


In [ ]:
def build_prompt(sentence, examples):
    """Return the prompt asking for sentence's translation, with the example pairs above it.

    TODO: write it. examples is a list of (English, translation) pairs, possibly empty. The
    prompt must say what to do (translate into LANGUAGE), show every example pair, and end with
    the sentence to translate. The wording is yours: it changes the result, and the write-up
    asks about it. Asking for "the translation only" keeps the model from explaining itself.
    """
    raise NotImplementedError("write build_prompt, then run this cell again")


# Checks on a case you can read: two marked example pairs, and none.
two = build_prompt("the cat sleeps", [("good morning", "EXAMPLE-A"), ("thank you", "EXAMPLE-B")])
none = build_prompt("the cat sleeps", [])
assert LANGUAGE in none and "the cat sleeps" in none, "the prompt must name the language and the sentence"
for text in ("good morning", "EXAMPLE-A", "thank you", "EXAMPLE-B"):
    assert text in two, f"an example pair is missing from the prompt: {text!r}"
assert "EXAMPLE-A" not in none, "with no examples, the prompt must have none"
assert two.rindex("the cat sleeps") > two.rindex("EXAMPLE-B"), "the sentence must come after the examples"

examples = list(zip(pool.src, pool.tgt))
print(build_prompt(test.src.iloc[0], examples[:5]))


## Step 3: the model

The default is `Qwen/Qwen2.5-1.5B-Instruct`: a small, open chat model that needs no Hugging Face
token and fits a Colab GPU. You may choose another from the
[Hugging Face Hub](https://huggingface.co/models) that was not trained for translation (not
NLLB, M2M100 or MarianMT); name it in the write-up. It decodes greedily, so the same prompt
always gives the same translation, and a change in BLEU comes from the context alone.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"     # <- or another chat model that is not MT-trained

tokenizer = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.float16 if device.type == "cuda" else torch.float32
).to(device).eval()


def translate(sentences, examples, batch_size=8):
    """Translate each sentence with the same examples in its prompt; greedy decoding."""
    out = []
    for start in range(0, len(sentences), batch_size):
        prompts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": build_prompt(s, examples)}],
                tokenize=False, add_generation_prompt=True,
            )
            for s in sentences[start:start + batch_size]
        ]
        batch = tokenizer(prompts, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            ids = model.generate(**batch, max_new_tokens=128, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        for row in ids[:, batch["input_ids"].shape[1]:]:
            text = tokenizer.decode(row, skip_special_tokens=True).strip()
            out.append(text.splitlines()[0].strip() if text else "")
    return out


print(f"{MODEL} on {device}: {sum(p.numel() for p in model.parameters()):,} parameters")


## Step 4: predict, then translate at each context size

**Write a prediction first**, in `PREDICTION`: which size will score highest, and why. The outer
loop is over context sizes, the inner over the test sentences; each size saves its pairs,
`context-k.tsv`, and its translations, `hyp-k.txt`.


In [ ]:
from torchlingo.evaluation import compute_bleu

# Before you run this: which context size will score highest for your language, and why?
# One or two sentences. The write-up asks you to reconcile it with what happens.
PREDICTION = ""
if not PREDICTION.strip():
    print("No prediction written. Write one before you run this; the report will say it is missing.\n")

CONTEXT_SIZES = [0, 5, 10, 20]
sources, references = list(test.src), list(test.tgt)

results = []
for k in CONTEXT_SIZES:
    pool.iloc[:k].to_csv(OUT_DIR / f"context-{k}.tsv", sep="\t", index=False)
    hyps = translate(sources, examples[:k])
    (OUT_DIR / f"hyp-{k}.txt").write_text("\n".join(hyps) + "\n", encoding="utf-8")
    bleu = compute_bleu(hyps, references)
    results.append({"context": k, "bleu": round(bleu.score, 2), "signature": str(bleu.signature)})
    print(f"k = {k:2}   BLEU {bleu.score:6.2f}   e.g. {hyps[0]!r}")

print(f"\nreference for that sentence: {references[0]!r}")


## Step 5: the report and the chart

The table is the assignment's BLEU report, saved as `bleu-report.tsv` with its signature. Fifty
sentences is a small test set: a point or two between sizes may be noise.


In [ ]:
import matplotlib.pyplot as plt

report = pd.DataFrame(results)
report.insert(0, "model", MODEL)
report.insert(0, "language", LANGUAGE)
report.to_csv(OUT_DIR / "bleu-report.tsv", sep="\t", index=False)
print(report[["context", "bleu"]].to_string(index=False))
print(f"signature: {results[0]['signature']}")
print(f"your prediction: {PREDICTION.strip() or '(none written)'}")

plt.plot(report.context, report.bleu, marker="o")
plt.xticks(CONTEXT_SIZES)
plt.xlabel("example pairs in the prompt")
plt.ylabel("BLEU")
plt.title(f"{LANGUAGE}, {MODEL}")
plt.savefig(OUT_DIR / "bleu-by-context.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nUpload from {OUT_DIR}: context-0/5/10/20.tsv, test.tsv and bleu-report.tsv.")


## Step 6: analysis

Answer these in your write-up, with which model you used, your prediction from Step 4, and one
sentence reconciling it with your result:

1. **What was the best context size for your language? Did you see a point where adding more
   context gave diminishing returns, or made translation worse?**
2. **Why do you think in-context examples help (or hurt) translation quality? Explain the possible
   mechanisms.**
3. **What were the main challenges you met? (Model choice, prompt wording, compute, and so on.)**

**Upload** to your folder in the Assignment shared folder: the four `context-k.tsv` files,
`test.tsv`, `bleu-report.tsv` and the write-up.
